# GAFIME v1 Practice Notebook

This notebook uses bounded deterministic data and the public API. It probes capabilities before running continuous, compiled, time-series, decision-path, and selector examples. For every public parameter, lifecycle, result field, and compatibility surface, use [the authoritative v1 API reference](https://github.com/onlyxItachi/GAFIME/blob/main/docs/notebooks/gafime_v1_api_reference.ipynb).


## 1. Version and Capability Probe


In [ ]:
import gafime
from gafime import backend_capabilities

print('GAFIME', gafime.__version__)
caps = backend_capabilities('auto', probe=True, precision='mixed')
print('configured:', caps.configured_backend)
print('selected:', caps.selected_backend)
print('status:', caps.selection_status)
print('device:', caps.device.value)


## 2. Deterministic Practice Data


In [ ]:
X = [
    [float(i), float((i * 7) % 11), float((i % 5) - 2)]
    for i in range(64)
]
y = [0.4 * row[0] * row[1] - 0.2 * row[2] for row in X]
feature_names = ['trend', 'cycle', 'offset']
len(X), len(X[0])


## 3. Reproducible Core Analysis


In [ ]:
from gafime import ComputeBudget, EngineConfig, GafimeEngine
config = EngineConfig(
    metric_names=('pearson', 'r2'),
    backend='core',
    precision='mixed',
    permutation_tests=0,
    num_repeats=1,
    budget=ComputeBudget(
        max_comb_size=2, max_combinations_per_k=64
    ),
)
report = GafimeEngine(config).analyze(X, y, feature_names)
print(report.backend.selected_backend, report.backend.execution_placement)
list(report.interactions.top_k(5, metric_name='pearson'))


## 4. Auto Backend Selection


In [ ]:
from dataclasses import replace

auto_config = replace(config, backend='auto')
auto_report = GafimeEngine(auto_config).analyze(X, y, feature_names)
print('selected:', auto_report.backend.selected_backend)
print('warnings:', auto_report.warnings)


## 5. Explicit Compiled Artifact


In [ ]:
from gafime import CompileFlags, compile
artifact = compile(
    X, y, feature_names, config=config,
    flags=CompileFlags(export=True),
)
try:
    compiled_report = artifact.analyze()
    print('compiled backend:', artifact.backend)
    print(compiled_report.interactions.top_k(2, 'pearson'))
finally:
    artifact.close()


## 6. Time-Series Generated Family


In [ ]:
time_series_config = EngineConfig(
    backend='core',
    precision='mixed',
    metric_names=('pearson', 'r2'),
    enable_time_series_functions=True,
    time_series_lags=(1, 2),
    time_series_windows=(4,),
    permutation_tests=0,
    num_repeats=1,
    budget=ComputeBudget(
        max_comb_size=1,
        max_combinations_per_k=128,
        max_time_series_candidates=32,
        top_k_features_for_time_series=3,
    ),
)
time_report = GafimeEngine(time_series_config).analyze(
    X, y, feature_names
)
print(time_report.warnings)
list(time_report.interactions.top_k(5, metric_name='pearson'))


Time-series lags and windows use the supplied row order. Sort input first and partition entity groups outside GAFIME so windows do not cross group boundaries.


## 7. Decision-Path Generated Family


In [ ]:
decision_config = EngineConfig(
    backend='core',
    precision='mixed',
    metric_names=('pearson', 'r2'),
    enable_decision_path_functions=True, permutation_tests=25,
    decision_path_max_depth=2,
    decision_path_max_paths=8,
    decision_path_min_leaf=4,
    num_repeats=1,
    budget=ComputeBudget(max_comb_size=1),
)
decision_report = GafimeEngine(decision_config).analyze(
    X, y, feature_names
)
print(decision_report.warnings)
list(decision_report.interactions.top_k(5, metric_name='pearson'))


Decision-path permutation maxT rediscovers paths for every permuted target before rescoring the full expanded family. Bootstrap stability resamples an already-selected candidate on the same rows; it measures variability conditional on selection, not out-of-sample generalization.


## 8. Family Capability Disclosure


In [ ]:
from gafime import available_families

for family in available_families():
    significance = family.significance_support
    print(
        family.name,
        'generation=', family.generation_placement,
        'scoring=', family.scoring_backends,
        'permutation=', significance.permutation,
        'stability=', significance.stability,
    )


## 9. sklearn-Style Pair Selection


In [ ]:
from gafime import GafimeSelector
selector = GafimeSelector(k=1, metric='pearson', precision='mixed')
augmented = selector.fit_transform(X, y)
print('selected pairs:', selector.top_interactions_)
print('shape:', len(augmented), 'x', len(augmented[0]))


For model evaluation, place `GafimeSelector` inside a scikit-learn Pipeline so discovery is refit on every training fold. Install the optional integration once beta.2 is published with `python -m pip install "gafime[sklearn]==1.0.0b2"`.
